# ComicAnalizer - Magi + PaddleOCR Pipeline

Este notebook genera la salida base para trabajar: detecciones Magi, reporte de calidad y comparacion OCR complementaria con PaddleOCR.

Antes de correrlo, activa GPU en Colab: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!rm -rf /content/ComicAnalizer
!git clone https://github.com/nicolas4432/ComicAnalizer.git /content/ComicAnalizer
%cd /content/ComicAnalizer
!git log --oneline -5

In [ ]:

%cd /content/ComicAnalizer

import subprocess, sys, textwrap

commands = [
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers==4.49.0',
        'huggingface_hub<1.0',
        'timm',
        'einops',
        'pytorch-metric-learning',
        'shapely',
        'setuptools',
    ],
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'paddlepaddle==3.1.1',
        '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cpu/',
    ],
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'paddleocr==3.5.0',
    ],
]

for cmd in commands:
    print('Instalando:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('Verificando PaddleOCR...')
import paddle
import paddleocr
print('paddle:', paddle.__version__)
print('paddleocr:', getattr(paddleocr, '__version__', 'unknown'))
print('paddle cuda disponible:', paddle.is_compiled_with_cuda())


## Subir dataset limpio

Para la prueba completa sube `magi_clean_full.zip`, generado localmente desde `outputs/packages/magi_clean_full.zip`.

El ZIP debe contener `by_comic/<comic_id>/test_1_clean/*.jpg`. El notebook detecta el nombre del paquete automaticamente desde el ZIP que subes.


In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
PACKAGE_NAME = zip_name.replace('.zip', '')
print('ZIP subido:', zip_name)
print('PACKAGE_NAME detectado:', PACKAGE_NAME)

!rm -rf /content/magi_sample
!mkdir -p /content/magi_sample
!unzip -q -o "$zip_name" -d /content/magi_sample
!find /content/magi_sample -maxdepth 5 -type d | head -40


## Ejecutar Magi

Por defecto procesa todos los comics limpios del ZIP. Para limitar a un comic especifico, define `COMIC_ID = 'nekkorarekko'`. Para todos los comics, deja `COMIC_ID = ''`.


In [ ]:

RUN_NAME = 'colab_clean_full'
DATASET_NAME = 'test_1_clean'
COMIC_ID = ''  # usa 'nekkorarekko' u otro comic_id para filtrar; '' procesa todos
MAX_COMICS = 0  # 0 = todos los comics del ZIP

RUN_ROOT = f'outputs/runs/{RUN_NAME}'
MAGI_OUTPUT = f'{RUN_ROOT}/magi'
MAGI_VISUALS = f'{RUN_ROOT}/visuals/magi_boxes'
OCR_VISUALS = f'{RUN_ROOT}/visuals/ocr_boxes'
ANALYSIS_OUTPUT = f'{RUN_ROOT}/analysis/magi_analysis_report.json'
OCR_OUTPUT = f'{RUN_ROOT}/analysis/paddle_magi_ocr_comparison.json'
OCR_EVIDENCE_OUTPUT = f'{RUN_ROOT}/analysis/ocr_evidence'
MAGI_CACHE = 'outputs/cache/magi'
IMAGE_ROOT = f'/content/magi_sample/{PACKAGE_NAME}/by_comic'

import shlex
import subprocess
from pathlib import Path


def module_help(module_name):
    result = subprocess.run(
        ['python', '-m', module_name, '--help'],
        text=True,
        capture_output=True,
    )
    return (result.stdout or '') + (result.stderr or '')


def supports_flag(help_text, flag):
    return flag in help_text


help_text = module_help('tools.inspect_magi_dataset')
cmd = [
    'python', '-m', 'tools.inspect_magi_dataset',
    '--input', IMAGE_ROOT,
    '--output-dir', MAGI_OUTPUT,
    '--all-pages-per-comic',
    '--dataset-name', DATASET_NAME,
    '--task', 'detections',
    '--cache-dir', MAGI_CACHE,
    '--device', 'cuda',
    '--dtype', 'float16',
    '--max-comics', str(MAX_COMICS),
]

if supports_flag(help_text, '--visual-output-dir'):
    cmd.extend(['--visual-output-dir', MAGI_VISUALS])
else:
    print('Aviso: esta version no soporta --visual-output-dir; los overlays Magi quedaran dentro de MAGI_OUTPUT.')

if supports_flag(help_text, '--no-panel-crops'):
    cmd.append('--no-panel-crops')
else:
    print('Aviso: esta version no soporta --no-panel-crops; puede generar recortes de paneles.')

if COMIC_ID and supports_flag(help_text, '--comic-id'):
    cmd.extend(['--comic-id', COMIC_ID])
elif COMIC_ID:
    print('Aviso: esta version no soporta --comic-id. Si el ZIP contiene solo ese comic, no hay problema.')

print('Ejecutando Magi:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)


## Generar reporte normalizado de calidad

In [ ]:
!python -m tools.analyze_magi_results \
  --input $MAGI_OUTPUT \
  --output $ANALYSIS_OUTPUT \
  --top-n 20

In [ ]:
import json
from pathlib import Path

report = json.loads(Path(ANALYSIS_OUTPUT).read_text())
print(json.dumps(report['summary'], indent=2, ensure_ascii=False))
print('\nPaginas sospechosas:', report['summary']['suspicious_page_count'])
print('\nTop flags:', report['summary']['flag_counts'])

## Comparar Magi Contra PaddleOCR

`OCR_LIMIT = 12` hace una muestra aleatoria razonable. Usa `OCR_LIMIT = 0` solo si quieres OCR en todas las paginas; puede tardar bastante segun runtime/GPU/CPU.


In [ ]:

OCR_LIMIT = 12  # 12 = muestra rapida; 0 = todas las paginas seleccionadas
OCR_SELECTION = 'random'  # random, first, suspicious

import shlex
import subprocess

help_text = module_help('tools.compare_magi_paddleocr')
cmd = [
    'python', '-m', 'tools.compare_magi_paddleocr',
    '--magi-input', MAGI_OUTPUT,
    '--image-root', IMAGE_ROOT,
    '--dataset-name', DATASET_NAME,
    '--selection', OCR_SELECTION,
    '--limit', str(OCR_LIMIT),
    '--seed', '42',
    '--lang', 'en',
    '--output', OCR_OUTPUT,
]

if supports_flag(help_text, '--visual-output-dir'):
    cmd.extend(['--visual-output-dir', OCR_VISUALS])
else:
    print('Aviso: esta version no soporta --visual-output-dir para OCR; solo guardara JSON.')

if COMIC_ID and supports_flag(help_text, '--comic-id'):
    cmd.extend(['--comic-id', COMIC_ID])
elif COMIC_ID:
    print('Aviso: esta version no soporta --comic-id en OCR. Si el ZIP contiene solo ese comic, no hay problema.')

print('Verificando import PaddleOCR antes de OCR...')
subprocess.run([
    'python', '-c',
    'import paddle, paddleocr; print(\"paddle=\", paddle.__version__); print(\"paddleocr=\", getattr(paddleocr, \"__version__\", \"unknown\"))'
], check=True)

print('Ejecutando comparacion PaddleOCR:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)


In [ ]:
ocr_report = json.loads(Path(OCR_OUTPUT).read_text())
print(json.dumps(ocr_report['summary'], indent=2, ensure_ascii=False))
for item in ocr_report['comparisons'][:10]:
    print(item['comic_id'], item['file_name'], 'Magi=', item['magi_text_regions'], 'Paddle=', item['paddle_text_blocks'], 'match=', item['matched_regions'], 't=', round(item['paddle_elapsed_seconds'], 2))

## Exportar Evidencia OCR

Convierte los resultados OCR en crops, overlays, metadata y plantillas de correccion. Se ejecuta sobre las paginas que pasaron por PaddleOCR.


In [ ]:

EXPORT_OCR_EVIDENCE = True

if EXPORT_OCR_EVIDENCE and Path(OCR_OUTPUT).exists():
    cmd = [
        'python', '-m', 'tools.export_ocr_evidence',
        '--ocr-report', OCR_OUTPUT,
        '--magi-input', MAGI_OUTPUT,
        '--output-dir', OCR_EVIDENCE_OUTPUT,
    ]
    print('Exportando evidencia OCR:')
    print(' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)
else:
    print('Sin evidencia OCR: no existe OCR_OUTPUT o EXPORT_OCR_EVIDENCE=False')


## Descargar salida estandar

In [ ]:

from google.colab import files

zip_out = f'{RUN_NAME}_magi_ocr_outputs.zip'
!zip -qr "$zip_out" \
  $RUN_ROOT \
  $MAGI_CACHE

files.download(zip_out)
